# Agentic KYC/AML Compliance System — Walkthrough

This notebook demonstrates the full LangGraph pipeline end-to-end on three scenarios:

| Scenario | What it exercises | Expected outcome |
|----------|-------------------|------------------|
| **A. Clean customer** | happy path | `LOW` → auto-approve |
| **B. Sanctions match** | identity screening + human-in-the-loop | `HIGH` → SAR draft → analyst review |
| **C. Suspicious transactions** | ML transaction monitoring + HITL | `HIGH` → SAR draft → analyst review |

**Pipeline**

```mermaid
flowchart LR
    START --> intake --> screening --> monitoring --> risk_scoring
    risk_scoring -->|LOW| auto_approve --> END
    risk_scoring -->|MEDIUM| request_info --> END
    risk_scoring -->|HIGH| sar_draft --> human_review --> END
```

**Design choices**
- **Hybrid ML + LLM** — a trained LightGBM classifier scores transactions; the LLM only extracts, reasons, and drafts the SAR narrative.
- **Deterministic, explainable risk tiering** — no black-box scoring; every decision has a one-line reason in the audit log.
- **Human-in-the-loop** — `HIGH`-risk cases pause at `human_review` via a LangGraph `interrupt()` and resume with the analyst's verdict.

> Requires `GROQ_API_KEY` in `.env`, the trained model at `data/models/lgbm_aml.pkl`, and the datasets in `data/raw/`.

## 1. Setup

Add the project root to the path, load environment variables, and build the compiled graph.

In [1]:
import sys
from pathlib import Path

# Make the project importable whether run from repo root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from langgraph.types import Command

from src.config import MODEL_CHEAP, MODEL_STRONG
from src.data.generate_kyc_docs import doc_to_text
from src.graph import build_graph
from src.state import CaseState

print(f"extraction model: {MODEL_CHEAP}")
print(f"reasoning model:  {MODEL_STRONG}")

graph = build_graph()
print("graph nodes:", sorted(n for n in graph.get_graph().nodes if not n.startswith("__")))

extraction model: openai/gpt-oss-20b
reasoning model:  openai/gpt-oss-120b
graph nodes: ['auto_approve', 'human_review', 'intake', 'monitoring', 'request_info', 'risk_scoring', 'sar_draft', 'screening']


## 2. Helper: run a case and show its decision + audit trail

`run_case` renders a KYC document as free text, invokes the graph, and — if the case pauses at the human-in-the-loop `interrupt()` — resumes it with the analyst's verdict. It then prints the decision, risk tier, and the full audit log.

In [2]:
from IPython.display import Markdown, display


def run_case(customer_id: str, doc: dict, analyst_verdict: str = "APPROVED") -> dict:
    cfg = {"configurable": {"thread_id": customer_id}}
    result = graph.invoke(
        CaseState(customer_id=customer_id, raw_document=doc_to_text(doc)), cfg
    )

    if "__interrupt__" in result:
        print(f"  paused at human_review -> analyst resumes with: {analyst_verdict}\n")
        result = graph.invoke(Command(resume=analyst_verdict), cfg)

    ident = result.get("extracted_identity")
    print(f"decision   : {result.get('decision')}")
    print(f"risk_tier  : {result.get('risk_tier')}")
    print(f"extracted  : {getattr(ident, 'full_name', None)!r} "
          f"(account {getattr(ident, 'account_id', None)!r})")
    print(f"screening  : {len(result.get('screening_hits', []))} hit(s)")
    print(f"txn_risk   : {result.get('transaction_risk_score')}")
    print("\naudit trail:")
    for entry in result.get("audit_log", []):
        print(f"  - {entry.node:13s} {entry.summary}")

    if result.get("sar_narrative"):
        display(Markdown("**SAR narrative:**\n\n> " +
                         result["sar_narrative"].replace("\n", "\n> ")))
    return result

## 3. Scenario A — Clean customer (auto-approve)

A well-formed identity with no sanctions match and a low-risk account. The graph should extract cleanly, find no hits, score low, and **auto-approve**.

In [3]:
clean_customer = {
    "full_name": "Gregory Fairbanks",
    "date_of_birth": "1985-04-12",
    "nationality": "United States",
    "address": "42 Maple Street, Springfield",
    "id_number": "US123456",
    "account_id": "8000EBD30",  # low model risk
}

_ = run_case("case-clean", clean_customer)

decision   : AUTO_APPROVED
risk_tier  : LOW
extracted  : 'Gregory Fairbanks' (account '8000EBD30')
screening  : 0 hit(s)
txn_risk   : 0.03406984058193163

audit trail:
  - intake        extracted identity; 0 missing fields
  - screening     0 sanctions/PEP hits for 'Gregory Fairbanks'
  - monitoring    txn risk 0.03, 5 flagged
  - risk_scoring  tier=LOW
  - auto_approve  low risk, approved


## 4. Scenario B — Sanctions match (escalate to human)

The customer's name matches a real person on the OpenSanctions list. Screening produces a hit, risk is tiered `HIGH`, the LLM drafts a SAR narrative, and the graph **pauses** at the human-in-the-loop interrupt until an analyst decides.

In [4]:
sanctioned_customer = {
    "full_name": "SANAVBARI NIKITENKO",  # real OpenSanctions person
    "date_of_birth": "1970-02-02",
    "nationality": "Russia",
    "address": "12 Nevsky Prospekt, St Petersburg",
    "id_number": "RU998877",
    "account_id": "8000EBD30",
}

_ = run_case("case-sanctions", sanctioned_customer, analyst_verdict="APPROVED")

Deserializing unregistered type src.state.ScreeningHit from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.state', 'ScreeningHit')]


Deserializing unregistered type src.state.AuditEntry from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.state', 'AuditEntry')]


Deserializing unregistered type src.state.ExtractedIdentity from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('src.state', 'ExtractedIdentity')]


  paused at human_review -> analyst resumes with: APPROVED

decision   : HUMAN_APPROVED
risk_tier  : HIGH
extracted  : 'SANAVBARI NIKITENKO' (account '8000EBD30')
screening  : 1 hit(s)
txn_risk   : 0.03406984058193163

audit trail:
  - intake        extracted identity; 0 missing fields
  - screening     1 sanctions/PEP hits for 'SANAVBARI NIKITENKO'
  - monitoring    txn risk 0.03, 5 flagged
  - risk_scoring  tier=HIGH
  - sar_draft     SAR narrative drafted
  - human_review  analyst: APPROVED


**SAR narrative:**

> **Suspicious Activity Report – Narrative**
> 
> **Customer Profile**  
> - **Name:** SANAVBARI NIKITENKO  
> - **Date of Birth:** 02 Feb 1970  
> - **Nationality:** Russian  
> - **Address:** 12 Nevsky Prospekt, St Petersburg, Russia  
> - **Identification:** RU998877 (Russian passport/ID)  
> - **Account:** 8000EBD30  
> 
> **Screening Result**  
> - **Match:** 100 % on the name “SANAVBARI NIKITENKO”  
> - **Entity Type:** Sanctioned individual  
> - **Source:** INTERPOL Red Notices  
> - **Reference ID:** NK‑224TRezPqwzhQZ37exWxtX  
> - **Listed Countries:** Russia; Tajikistan  
> 
> The customer is a perfect‑match hit on an INTERPOL Red Notice, indicating a high‑risk sanctions exposure.
> 
> **Flagged Transaction Activity (September 2022)**  
> 
> | Timestamp (UTC) | Amount (USD) | Risk Score* |
> |------------------|--------------|------------|
> | 2022‑09‑02 16:25 | 145.14 | 0.0047 |
> | 2022‑09‑05 00:04 | 11.11  | 0.0022 |
> | 2022‑09‑05 22:29 | 146.66 | 0.0341 |
> | 2022‑09‑06 00:01 | 125.32 | 0.0023 |
> | 2022‑09‑09 14:06 | 145.14 | 0.0055 |
> 
> *Risk scores are generated by the transaction monitoring system; higher values indicate greater anomaly.
> 
> **Observations**  
> 1. The account holder is a confirmed match to a sanctioned individual (INTERPOL Red Notice).  
> 2. Within a one‑week window, the customer executed five outbound payments ranging from $11.11 to $146.66.  
> 3. The majority of the payments are clustered around $145‑$147, a pattern often associated with “structuring” or “smurfing” to keep individual transaction amounts below typical reporting thresholds while moving funds repeatedly.  
> 4. The highest risk score (0.034) corresponds to the 22:29 Sept 5 transaction, which is also the largest amount in the series.  
> 
> **Reason for Escalation**  
> The combination of a 100 % sanctions match and a series of repeated, similarly‑sized outbound payments suggests possible attempts to move funds on behalf of, or for the benefit of, a sanctioned individual. This activity meets the criteria for a potential sanctions violation and warrants immediate escalation to the sanctions compliance team and filing of a SAR with the appropriate authorities.  
> 
> **Recommended Action**  
> - Freeze or place a hold on the account pending further investigation.  
> - Conduct enhanced due‑diligence on the source of funds and the ultimate beneficiaries of the listed transactions.  
> - Submit a formal SAR to the relevant financial intelligence unit (FIU) and notify the sanctions compliance unit.  
> 
> ---  
> *Prepared by: Compliance Analyst (automated SAR generation)*  
> *Date: 28 Jul 2026*  

## 5. Scenario C — Suspicious transactions (escalate to human)

The identity is clean (no sanctions hit), but the linked account maps to transactions the trained LightGBM model flags as high-risk. The transaction risk drives the `HIGH` tier, and the case again escalates to human review.

In [5]:
suspicious_customer = {
    "full_name": "Marcus Wellington",
    "date_of_birth": "1979-09-30",
    "nationality": "United Kingdom",
    "address": "8 Baker Street, London",
    "id_number": "UK445566",
    "account_id": "100428660",  # tied to Is Laundering=1 transactions
}

_ = run_case("case-suspicious", suspicious_customer, analyst_verdict="ESCALATE")

  paused at human_review -> analyst resumes with: ESCALATE

decision   : HUMAN_ESCALATE
risk_tier  : HIGH
extracted  : 'Marcus Wellington' (account '100428660')
screening  : 0 hit(s)
txn_risk   : 1.0

audit trail:
  - intake        extracted identity; 0 missing fields
  - screening     0 sanctions/PEP hits for 'Marcus Wellington'
  - monitoring    txn risk 1.00, 5 flagged
  - risk_scoring  tier=HIGH
  - sar_draft     SAR narrative drafted
  - human_review  analyst: ESCALATE


**SAR narrative:**

> **Suspicious Activity Report – Narrative (Draft)**  
> 
> **Customer Information**  
> - **Name:** Marcus Wellington  
> - **Date of Birth:** 30 Sep 1979  
> - **Nationality:** United Kingdom  
> - **Residential Address:** 8 Baker Street, London  
> - **Identification Number:** UK445566  
> - **Account ID:** 100428660  
> 
> **Screening Results**  
> - No adverse hits were returned from sanctions, PEP, or watch‑list screening at the time of review.  
> 
> **Flagged Transactions**  
> 
> | Date & Time (UTC) | Amount (GBP) | Risk Score |
> |-------------------|--------------|------------|
> | 2022‑09‑01 07:22  | 303.54       | 1.0 |
> | 2022‑09‑06 10:01  | 295.08       | 1.0 |
> | 2022‑09‑07 06:26  | 304.21       | 1.0 |
> | 2022‑09‑09 13:41  | 287.80       | 1.0 |
> | 2022‑09‑10 06:53  | 305.80       | 1.0 |
> 
> **Facts**  
> 1. Over a ten‑day window (1 Sep 2022 – 10 Sep 2022), five outbound payments were made from the same account, each ranging from £287.80 to £305.80.  
> 2. All five transactions received the maximum internal risk rating (1.0) generated by the transaction monitoring system.  
> 3. The payments are clustered in time (average interval ≈ 2 days) and are of near‑identical amounts, a pattern commonly associated with structuring or “smurfing” to avoid detection thresholds.  
> 4. No corresponding inbound activity, business justification, or documented purpose for these payments was identified in the account’s transaction history or supporting documentation.  
> 
> **Reason for Escalation**  
> The combination of (a) multiple, similarly sized outbound payments within a short period, (b) the highest risk score assigned by the monitoring engine, and (c) the absence of a legitimate business or personal rationale raises suspicion of potential structuring, money‑laundering, or other illicit activity. Consequently, the case warrants escalation to the compliance team for a detailed investigation and, if warranted, filing of a formal SAR with the relevant authorities.  

## 6. Takeaways

- **Human-in-the-loop is the deployment blocker in regulated finance, not model quality.** High-risk cases triage to an analyst rather than auto-approving — the graph literally pauses via `interrupt()` and resumes with the human verdict.
- **Detection is a trained classifier; the LLM only reasons and drafts.** LLMs are the wrong tool for high-volume transaction scoring on cost and latency.
- **Every node appends to an immutable audit log,** so any decision is fully traceable — which is what regulators expect.

**Honest limitations:** the transaction model uses a handful of lightweight per-transaction features and has weak precision (PR-AUC near the base rate) — real detection needs account/graph-level features (velocity, fan-in/out, counterparty structure). Production would also need PII handling, real-time streaming, and model monitoring / drift detection.